# Kraken Batch OCR Notebook
This notebook runs Kraken batch OCR over the copied evaluation crops inside `Evaluation/Kraken`, saves one text file per crop, and assembles a TSV summary for inspection.

The workflow is model-driven: Kraken does not use a separate language flag for OCR. Recognition behavior depends on the `.mlmodel` you supply.

## 1. Install And Verify Kraken
Use the current Python environment for the notebook. If `kraken` is not available, switch this notebook to the `kraken-ocr` kernel before running the rest of the cells.

In [ ]:
from pathlib import Path
import shutil
import subprocess

KRAKEN_ENV = "kraken-ocr"
conda_candidates = [shutil.which("conda"), "/home/user/anaconda3/bin/conda"]
conda_bin = next((candidate for candidate in conda_candidates if candidate and Path(candidate).exists()), None)

if conda_bin is None:
    print("conda executable not found in the current environment.")
    print("Install or expose conda in PATH before running the Kraken batch cells.")
else:
    print("conda executable:", conda_bin)
    version = subprocess.run(
        [conda_bin, "run", "-n", KRAKEN_ENV, "kraken", "--version"],
        capture_output=True,
        text=True,
        check=False,
    )
    print(version.stdout.strip() or version.stderr.strip())

## 2. Import Python Helpers
These imports are used for file discovery, subprocess execution, timing, and tabular summaries.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import json
import os
import shutil
import subprocess
import time

import pandas as pd

## 3. Define Input And Output Paths
All paths point to the organized Kraken evaluation folder so the notebook is self-contained.

In [ ]:
CROPS_DIR = "path/to/your/crops/directory"  # Update this path to your actual crops directory
OUTPUT_DIR = "path/to/your/output_directory"  # Update this path to your actual output directory
TSV_PATH = "path/to/your/output_directory/kraken_dev_predictions.tsv"  # Update this path to your actual TSV file
STATUS_CSV_PATH = "path/to/your/output_directory/kraken_dev_batch_status.csv"  # Update this path to your actual status CSV file
MODEL_PATH = Path("path/to/your/model/file.mlmodel")  # Update this path to your actual model file
KRAKEN_ENV = "kraken-ocr"
CONDA_CANDIDATES = [shutil.which("conda"), "/home/user/anaconda3/bin/conda"]
CONDA_BIN = next((candidate for candidate in CONDA_CANDIDATES if candidate and Path(candidate).exists()), None)
KRAKEN_BASE_COMMAND = [CONDA_BIN, "run", "-n", KRAKEN_ENV, "kraken"] if CONDA_BIN else None

# Use "paragraph" for paragraph/page-like crops (segment + ocr).
# Use "line" for pre-segmented single-line crops (ocr -s).
OCR_INPUT_MODE = "paragraph"  # "paragraph" or "line"

MAX_IMAGES = None  # set an integer for a quick test run
RUN_PARALLEL = True
MAX_WORKERS = 4

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TSV_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Crops dir exists: {CROPS_DIR.exists()} -> {CROPS_DIR}")
print(f"Model exists: {MODEL_PATH.exists()} -> {MODEL_PATH}")
print(f"Conda exists: {CONDA_BIN is not None} -> {CONDA_BIN}")
print(f"Kraken env: {KRAKEN_ENV}")
print(f"OCR input mode: {OCR_INPUT_MODE}")
print(f"Output dir: {OUTPUT_DIR}")

## 4. Collect Image Files For Batch Processing
This builds a stable list of input crops so repeated notebook runs use the same ordering.

In [ ]:
image_files = sorted([*CROPS_DIR.rglob("*.jpg"), *CROPS_DIR.rglob("*.png")])
if MAX_IMAGES is not None:
    image_files = image_files[:MAX_IMAGES]

df_inputs = pd.DataFrame({
    "crop_path": [str(path) for path in image_files],
    "relative_path": [str(path.relative_to(CROPS_DIR)) for path in image_files],
})

print(f"Images queued: {len(df_inputs)}")
df_inputs.head(10)

## 5. Build Kraken Commands
Commands adapt to input type: `paragraph` mode uses layout analysis (`segment -bl ocr`), while `line` mode skips segmentation (`ocr -s`).

In [ ]:
def output_path_for(image_path: Path) -> Path:
    relative = image_path.relative_to(CROPS_DIR)
    return OUTPUT_DIR / relative.with_suffix(".txt")

def build_kraken_command(image_path: Path, output_path: Path) -> list[str]:
    if KRAKEN_BASE_COMMAND is None:
        raise RuntimeError("conda was not found, so the Kraken command cannot be constructed.")

    if OCR_INPUT_MODE == "line":
        pipeline = ["ocr", "-m", str(MODEL_PATH), "-s"]
    elif OCR_INPUT_MODE == "paragraph":
        pipeline = ["segment", "-bl", "ocr", "-m", str(MODEL_PATH)]
    else:
        raise ValueError(f"Unsupported OCR_INPUT_MODE: {OCR_INPUT_MODE}. Use 'line' or 'paragraph'.")

    return [
        *KRAKEN_BASE_COMMAND,
        "-i", str(image_path), str(output_path),
        *pipeline,
    ]

preview_commands = [
    {
        "crop_path": str(path),
        "output_path": str(output_path_for(path)),
        "command": " ".join(build_kraken_command(path, output_path_for(path))),
    }
    for path in image_files[:3]
    if KRAKEN_BASE_COMMAND is not None
]

pd.DataFrame(preview_commands)

## 6. Run Batch Kraken Jobs
This cell executes the batch. The subprocess call detaches stdin so Kraken does not consume the notebook loop input and stop early.

In [ ]:
def run_kraken_ocr(image_path: Path) -> dict:
    output_path = output_path_for(image_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    command = build_kraken_command(image_path, output_path)
    start_time = time.perf_counter()
    completed = subprocess.run(
        command,
        cwd=WORKSPACE,
        capture_output=True,
        text=True,
        check=False,
        stdin=subprocess.DEVNULL,
    )
    elapsed = time.perf_counter() - start_time
    return {
        "crop_path": str(image_path),
        "output_path": str(output_path),
        "status": "ok" if completed.returncode == 0 else "failed",
        "returncode": completed.returncode,
        "elapsed_sec": round(elapsed, 3),
        "stdout": completed.stdout,
        "stderr": completed.stderr,
    }

def run_batch_parallel(paths: list[Path], max_workers: int = 4) -> list[dict]:
    records = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_path = {executor.submit(run_kraken_ocr, path): path for path in paths}
        total = len(future_to_path)
        for idx, future in enumerate(as_completed(future_to_path), start=1):
            record = future.result()
            records.append(record)
            if idx == 1 or idx % 25 == 0 or idx == total:
                print(f"Processed {idx}/{total}: {Path(record['crop_path']).name} -> {record['status']}")
    return records

if RUN_PARALLEL:
    batch_records = run_batch_parallel(image_files, max_workers=MAX_WORKERS)
    batch_df = pd.DataFrame(batch_records).sort_values('crop_path').reset_index(drop=True)
    batch_df.to_csv(STATUS_CSV_PATH, index=False)
    batch_df.head(10)
else:
    print("Set RUN_PARALLEL = True in the paths cell if you want to use this mode.")

## 7. Capture Output And Errors
Failed rows keep stdout and stderr so you can inspect Kraken errors without leaving the notebook.

In [ ]:
if 'batch_df' not in globals():
    batch_df = pd.DataFrame(batch_records) if 'batch_records' in globals() else pd.DataFrame()

print(f"Batch rows: {len(batch_df)}")
if not batch_df.empty:
    display(batch_df[batch_df['status'] != 'ok'][['crop_path', 'status', 'returncode', 'stderr']].head(20))
else:
    print("Run the batch cell first.")

## 8. Inspect Generated Results
This cell reads the text outputs, normalizes whitespace, and builds a TSV that is easy to compare with ground truth later.

In [ ]:
def normalize_text(text: str) -> str:
    return " ".join(str(text).split())

def join_lines(lines: list[str]) -> str:
    if lines == []:
        return ""

    lines = [
        line
        for line in (
            line.strip()
            for line in lines
        )
        if line != ""
    ]

    if lines == []:
        return ""

    new_lines = []
    last_line_index = len(lines) - 1
    for i, line in enumerate(lines):
        new_lines.append(line)
        if (
            i != last_line_index and (
                line[-1] not in "-—"
                or (len(line) > 1 and line[-2] == " ")
            )
        ):
            new_lines.append(" ")

    return "".join(new_lines)

prediction_rows = []
for image_path in image_files:
    text_path = output_path_for(image_path)
    raw_text = text_path.read_text(encoding="utf-8") if text_path.exists() else ""
    raw_lines = raw_text.splitlines()

    if OCR_INPUT_MODE == "paragraph":
        pred_text = normalize_text(join_lines(raw_lines))
    else:
        pred_text = normalize_text(raw_text)

    prediction_rows.append({
        "crop_path": str(image_path),
        "output_path": str(text_path),
        "pred_text": pred_text,
        "exists": text_path.exists(),
    })

pred_df = pd.DataFrame(prediction_rows)
pred_df.to_csv(TSV_PATH, sep="\t", index=False)

print(f"Predictions written: {TSV_PATH}")
print(f"Existing text outputs: {int(pred_df['exists'].sum())}/{len(pred_df)}")
pred_df.head(10)

# Evaluation Steps

## 9. Load Ground Truth And Build Evaluation Table
This mirrors the Tesseract notebook mapping logic: read `ocr_data_v11_test.json`, reconstruct crop paths, and align Kraken predictions with ground-truth transcriptions.

In [ ]:
JSON_PATH = WORKSPACE / "mlt_data" / "ocr_data_v11_dev.json"

with JSON_PATH.open("r", encoding="utf-8") as f:
    test_data = json.load(f)

if isinstance(test_data, dict):
    records = test_data.get("data", [])
elif isinstance(test_data, list):
    records = test_data
else:
    records = []

gt_rows = []
for item in records:
    if not isinstance(item, dict):
        continue

    page = item.get("page", {})
    fname = page.get("page_fname")
    if not fname:
        continue

    base_id = Path(fname).stem
    doc_id = str(page.get("document_id") or base_id.split("-")[0])
    page_id = str(page.get("page_id") or base_id.split("-")[1])
    boxes = item.get("boxes", [])

    for box_idx, box_item in enumerate(boxes):
        crop_path = CROPS_DIR / doc_id / page_id / f"{base_id}_box{box_idx:04d}.jpg"
        gt_text = box_item.get("transcription", "") if isinstance(box_item, dict) else ""
        gt_rows.append({
            "page_fname": fname,
            "box_idx": box_idx,
            "crop_path": str(crop_path),
            "gt_text": gt_text,
        })

df_gt = pd.DataFrame(gt_rows).sort_values(["page_fname", "box_idx"]).reset_index(drop=True)
df_eval = df_gt.merge(
    pred_df[["crop_path", "pred_text"]],
    on="crop_path",
    how="left",
).copy()

df_eval["gt_text"] = df_eval["gt_text"].fillna("")
df_eval["pred_text"] = df_eval["pred_text"].fillna("")

print(f"GT rows: {len(df_gt)}")
print(f"Evaluation rows: {len(df_eval)}")
print(f"Rows with prediction text: {(df_eval['pred_text'].str.len() > 0).sum()}")
df_eval[["page_fname", "box_idx", "gt_text", "pred_text"]].head(10)

## 10. Compute CER Metrics (Kraken)
This section reuses the same CER approach as the Tesseract notebook: normalized text, Levenshtein edit distance, weighted CER, and per-crop CER.

In [ ]:
import unicodedata

def normalize_text_cer(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s))
    s = " ".join(s.split())
    return s

def levenshtein_distance(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            ins = curr[j - 1] + 1
            dele = prev[j] + 1
            sub = prev[j - 1] + (ca != cb)
            curr.append(min(ins, dele, sub))
        prev = curr
    return prev[-1]

df_eval["gt_norm"] = df_eval["gt_text"].map(normalize_text_cer)
df_eval["pred_norm"] = df_eval["pred_text"].map(normalize_text_cer)
df_eval["gt_len"] = df_eval["gt_norm"].str.len()
df_eval["edit_distance"] = [
    levenshtein_distance(pred, gt)
    for pred, gt in zip(df_eval["pred_norm"], df_eval["gt_norm"])
]
df_eval["cer"] = df_eval["edit_distance"] / df_eval["gt_len"].replace(0, pd.NA)

weighted_cer = df_eval["edit_distance"].sum() / max(df_eval["gt_len"].sum(), 1)
mean_cer = df_eval["cer"].dropna().mean()

print(f"Evaluated crops: {len(df_eval)}")
print(f"Weighted CER: {weighted_cer:.4f}")
print(f"Mean CER (per-crop): {mean_cer:.4f}")

df_eval[["page_fname", "box_idx", "gt_len", "edit_distance", "cer", "gt_text", "pred_text"]].sort_values("cer", ascending=False).reset_index(drop=True).head(25)

In [ ]:
N_SHOW = 12
SNIPPET_LEN = 220

preview_df = df_eval.copy()
preview_df["gt_preview"] = preview_df["gt_text"].fillna("").str.slice(0, SNIPPET_LEN)
preview_df["pred_preview"] = preview_df["pred_text"].fillna("").str.slice(0, SNIPPET_LEN)

preview_df = preview_df.sort_values("cer", ascending=False).reset_index(drop=True)
top_hard = preview_df.head(N_SHOW // 2)
top_easy = preview_df.tail(N_SHOW - len(top_hard))
sample_preview = pd.concat([top_hard, top_easy], ignore_index=True)

sample_preview[["page_fname", "box_idx", "cer", "gt_preview", "pred_preview"]]

## 11. Page-Level Analysis And Export
This matches the page-level diagnostics and Excel-friendly export style used in the Tesseract notebook.

In [ ]:
PAGE_ID = "0060-015"  # e.g. 0003-003 or 0003-003.jpg
MAX_SHOW = None  # Set to None to show all crops for that page

target_page = PAGE_ID if PAGE_ID.endswith(".jpg") else f"{PAGE_ID}.jpg"
page_rows = df_eval[df_eval["page_fname"] == target_page].sort_values("box_idx").reset_index(drop=True)

if MAX_SHOW is not None:
    page_rows = page_rows.head(MAX_SHOW)

print(f"Page: {target_page}")
print(f"Crops shown: {len(page_rows)}")

if page_rows.empty:
    print("No rows found. Check PAGE_ID and ensure OCR/CER cells were run first.")
else:
    total_edits = page_rows["edit_distance"].sum()
    total_gt_len = int(page_rows["gt_len"].sum())
    page_weighted_cer = total_edits / max(total_gt_len, 1)
    page_mean_cer = page_rows["cer"].dropna().mean()

    print(f"Weighted CER for page: {page_weighted_cer:.4f}")
    print(f"Mean CER for page (per-crop): {page_mean_cer:.4f}")
    page_rows[["box_idx", "cer", "gt_text", "pred_text"]].head(12)

In [ ]:
def _clean_text(v):
    if v is None:
        return ""
    return " ".join(str(v).split())

if "page_rows" in globals() and page_rows is not None and len(page_rows) > 0:
    src = page_rows.copy()
else:
    src = df_eval.copy()

needed = [c for c in ["page_fname", "box_idx", "gt_text", "pred_text"] if c in src.columns]
out = src[needed].copy()

if "page_fname" not in out.columns:
    out["page_fname"] = ""
if "box_idx" not in out.columns:
    out["box_idx"] = range(len(out))

out = out.sort_values(["page_fname", "box_idx"], kind="stable")

export_df = out.assign(
    value=out.apply(lambda r: f"page:{_clean_text(r.get('page_fname', ''))} bb:{int(r.get('box_idx', 0)):04d}", axis=1),
    GT=out["gt_text"].map(_clean_text),
    Prediction=out["pred_text"].map(_clean_text),
)[["value", "GT", "Prediction"]]

KRAKEN_EVAL_TSV_PATH = WORKSPACE / "outputs" / "kraken_eval_page_or_all.tsv"
export_df.to_csv(KRAKEN_EVAL_TSV_PATH, sep="\t", index=False)

# print(f"Evaluation TSV written: {KRAKEN_EVAL_TSV_PATH}")
# export_df.head(20)

# Print as tab-separated values so copy/paste lands in Excel columns.
print(export_df.to_csv(sep="\t", index=False))

In [ ]:
from IPython.display import display, Markdown

page_summary = (
    df_eval.groupby("page_fname", dropna=False)
    .agg(
        total_edits=("edit_distance", "sum"),
        total_gt_len=("gt_len", "sum"),
        mean_cer=("cer", "mean"),
        num_boxes=("box_idx", "count"),
    )
    .reset_index()
)
page_summary["weighted_cer"] = page_summary["total_edits"] / page_summary["total_gt_len"].clip(lower=1)
page_summary = page_summary.sort_values(["weighted_cer", "mean_cer", "page_fname"], ascending=[False, False, True]).reset_index(drop=True)

worst_page = str(page_summary.loc[0, "page_fname"])
worst_page_rows = df_eval[df_eval["page_fname"] == worst_page].sort_values("box_idx").reset_index(drop=True)
worst_page_weighted_cer = float(page_summary.loc[0, "weighted_cer"])
worst_page_mean_cer = float(page_summary.loc[0, "mean_cer"])
worst_page_boxes = int(page_summary.loc[0, "num_boxes"])

print(f"Worst page by weighted CER: {worst_page}")
print(f"Crops shown: {worst_page_boxes}")
print(f"Weighted CER for page: {worst_page_weighted_cer:.4f}")
print(f"Mean CER for page (per-crop): {worst_page_mean_cer:.4f}")
print("=" * 80)

for _, row in worst_page_rows.iterrows():
    crop_path = Path(row["crop_path"])

    # display(Markdown(f"### box {int(row['box_idx']):04d} | CER: {float(row['cer']):.4f}"))
    # display(Image.open(crop_path))
    # display(Markdown("**GT**"))
    # print(str(row["gt_text"]))
    # display(Markdown("**Prediction**"))
    # print(str(row["pred_text"]))
    # print("-" * 80)